In [5]:
import random

class SimpleReflexVacuumAgent:
    def __init__(self, grid_size=2, steps=20):
        self.grid_size = grid_size
        self.steps = steps

        self.environment = [[random.choice([0, 1]) for _ in range(grid_size)]
                            for _ in range(grid_size)]

        self.x = random.randint(0, grid_size - 1)
        self.y = random.randint(0, grid_size - 1)

        self.rooms_cleaned = 0
        self.movements = 0

        self.directions = {
            "Up": (-1, 0),
            "Down": (1, 0),
            "Left": (0, -1),
            "Right": (0, 1)
        }

    def perceive(self):
        return (self.x, self.y, self.environment[self.x][self.y])

    def act(self):
        x, y, status = self.perceive()

        if status == 1: 
            self.environment[x][y] = 0
            self.rooms_cleaned += 1
        else:
            self.move_randomly()

    def move_randomly(self):
        valid_moves = []

        for dx, dy in self.directions.values():
            nx, ny = self.x + dx, self.y + dy
            if 0 <= nx < self.grid_size and 0 <= ny < self.grid_size:
                valid_moves.append((nx, ny))

        self.x, self.y = random.choice(valid_moves)
        self.movements += 1

    def run(self):
        for _ in range(self.steps):
            self.act()

    def report(self):
        print("Rooms cleaned:", self.rooms_cleaned)
        print("Total movements:", self.movements)


agent = SimpleReflexVacuumAgent()
agent.run()
agent.report()


Rooms cleaned: 2
Total movements: 18


In [6]:
import random

class ModelBasedVacuumAgent:
    def __init__(self, grid_size=2):
        self.grid_size = grid_size

        self.environment = [[random.choice([0, 1]) for _ in range(grid_size)]
                            for _ in range(grid_size)]

        self.model = [[None for _ in range(grid_size)]
                      for _ in range(grid_size)]

        self.x = random.randint(0, grid_size - 1)
        self.y = random.randint(0, grid_size - 1)

        self.rooms_cleaned = 0
        self.movements = 0

        self.directions = {
            "Up": (-1, 0),
            "Down": (1, 0),
            "Left": (0, -1),
            "Right": (0, 1)
        }

    def perceive(self):
        """Return current percept"""
        return self.x, self.y, self.environment[self.x][self.y]

    def update_model(self, x, y, status):
        """Update internal model based on percept"""
        self.model[x][y] = status

    def all_rooms_clean(self):
        """Check if all rooms are known and clean"""
        for row in self.model:
            for cell in row:
                if cell != 0:
                    return False
        return True

    def choose_move(self):
        """Prefer unvisited or dirty rooms"""
        candidates = []
        fallback = []

        for dx, dy in self.directions.values():
            nx, ny = self.x + dx, self.y + dy
            if 0 <= nx < self.grid_size and 0 <= ny < self.grid_size:
                if self.model[nx][ny] is None or self.model[nx][ny] == 1:
                    candidates.append((nx, ny))
                else:
                    fallback.append((nx, ny))

        return random.choice(candidates if candidates else fallback)

    def act(self):
        x, y, status = self.perceive()

        self.update_model(x, y, status)

        if status == 1:  # Dirty
            self.environment[x][y] = 0
            self.update_model(x, y, 0)
            self.rooms_cleaned += 1
        else:
            nx, ny = self.choose_move()
            self.x, self.y = nx, ny
            self.movements += 1

    def run(self):
        """Run until all rooms are known to be clean"""
        while not self.all_rooms_clean():
            self.act()

    def report(self):
        print("Rooms cleaned:", self.rooms_cleaned)
        print("Total movements:", self.movements)


# Run simulation
agent = ModelBasedVacuumAgent()
agent.run()
agent.report()


Rooms cleaned: 3
Total movements: 3
